# AnyCam — 01 · Download the extra datasets (run ONCE)

This notebook downloads DeepHS Fruit (papaya, kaki, mango) and the Estrada leaf dataset
straight from their websites into Kaggle, so nothing goes through your laptop.

**Before running:** in the right panel, **Settings → Internet → On**
(needs a phone-verified Kaggle account). GPU is NOT needed here.

**After it finishes:** click **Save Version → Save & Run All**. When the version is done, open it,
go to the **Output** tab and click **New Dataset**, name it `anycam-extra-data`, keep it **Private**.

In [ ]:
import os, subprocess, zipfile, shutil, pathlib
OUT = pathlib.Path('/kaggle/working/data')
OUT.mkdir(parents=True, exist_ok=True)
print(shutil.disk_usage('/kaggle/working'))

In [ ]:
BASE = 'https://cogsys.cs.uni-tuebingen.de/webprojects/DeepHS-Fruit-2023-Datasets/'
FILES = ['readme.txt', 'annotations.zip', 'Papaya.zip', 'Kaki.zip', 'Mango.zip']
dst = OUT / 'deephs_fruit'
dst.mkdir(exist_ok=True)
for f in FILES:
    target = dst / f
    if not target.exists():
        print('downloading', f)
        subprocess.run(['wget', '-q', '--show-progress', '-O', str(target), BASE + f], check=True)
    print(f, round(target.stat().st_size / 1e9, 2), 'GB')
    print('free disk:', round(shutil.disk_usage('/kaggle/working').free / 1e9, 1), 'GB')

In [ ]:
# unzip each archive into its own folder, then delete the zip to save space
for z in sorted(dst.glob('*.zip')):
    folder = dst / z.stem
    print('unzipping', z.name)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(folder)
    z.unlink()
    print('free disk:', round(shutil.disk_usage('/kaggle/working').free / 1e9, 1), 'GB')

In [ ]:
# Estrada et al. 2025 (avocado, olive, grape leaves; spectrometer + real MicaSense)
# Figshare 'download all' link for article 26950660, version 2
est = OUT / 'estrada_leaves'
est.mkdir(exist_ok=True)
url = 'https://figshare.com/ndownloader/articles/26950660/versions/2'
z = est / 'estrada_all.zip'
r = subprocess.run(['wget', '-q', '--show-progress', '-O', str(z), url])
if r.returncode == 0 and z.stat().st_size > 0:
    with zipfile.ZipFile(z) as zf:
        zf.extractall(est)
    z.unlink()
    print('Estrada OK')
else:
    print('Estrada download failed: download it by hand from https://doi.org/10.6084/m9.figshare.26950660.v2')

In [ ]:
# summary: what is in the output
for p in sorted(OUT.rglob('*'))[:200]:
    if p.is_dir() and p.relative_to(OUT).as_posix().count('/') < 3:
        n = sum(1 for _ in p.rglob('*') if _.is_file())
        print(p.relative_to(OUT), '-', n, 'files')
print('total size:', round(sum(f.stat().st_size for f in OUT.rglob('*') if f.is_file()) / 1e9, 2), 'GB')